In [ ]:
!nvidia-smi    

In [ ]:
!pip install -q uv

In [ ]:
!uv pip install --system --no-cache faiss-gpu pyarrow "huggingface_hub[cli]" bm25s rank-bm25 pyvi jieba loguru typer pydantic pyyaml tqdm flagembedding sentence-transformers trafilatura httpx qdrant_client

In [ ]:
import os                                                                                                                                                                             
from pathlib import Path                                                                                                                                                              
                                                                                                                                                                                      
REPO_URL = "https://github.com/neiths/med-doc-retrieval.git"                                                                                                                          
PROJECT_DIR = Path("/kaggle/working/med-doc-retrieval")                                                                                                                               
                                                                                                                                                                                      
if not PROJECT_DIR.exists():                                                                                                                                                          
    print("Cloning repository...")                                                                                                                                                    
    !git clone -b main $REPO_URL $PROJECT_DIR                                                                                                                                         
else:                                                                                                                                                                                 
    print("Updating repository...")                                                                                                                                                   
    %cd $PROJECT_DIR                                                                                                                                                                  
    !git pull origin main                                                                                                                                                             
                                                                                                                                                                                      
%cd $PROJECT_DIR                                                                                                                                                                      
print("Current Working Directory:", os.getcwd()) 

In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

api = HfApi(token=HF_TOKEN)

# 1. Tạo sẵn thư mục
os.makedirs("data/indices", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# 2. Danh sách file cần thiết cho Pure Dense Retrieval
all_files = [
    ("indices/dense_index.faiss", "data/indices/dense_index.faiss"),
    ("indices/chunks_meta.sqlite", "data/indices/chunks_meta.sqlite"),
    ("indices/dense_metadata.json", "data/indices/dense_metadata.json"),
    ("processed/pubmed_cache.jsonl", "data/processed/pubmed_cache.jsonl"),
]

# 3. Lọc ra các file chưa có hoặc dung lượng bằng 0
missing_files = []
for remote, local in all_files:
    p = Path(local)
    if not p.exists() or p.stat().st_size == 0:
        missing_files.append((remote, local))
    else:
        print(f"✓ Đã có sẵn: {local} ({p.stat().st_size / (1024*1024):.2f} MB)")

# 4. Tải các file còn thiếu
if missing_files:
    print(f"
Đang tải {len(missing_files)} file còn thiếu từ Hugging Face Bucket...")
    api.download_bucket_files(
        bucket_id="nieths/ViBioMIR",
        files=missing_files,
        token=HF_TOKEN,
    )
    print("✅ Tải xong toàn bộ các file còn thiếu!")
else:
    print("
✅ Tất cả các file đã có đầy đủ trên máy!")

In [ ]:
!ls -lh data/indices

In [ ]:
!HF_HOME=/tmp/hf python scripts/run_inference_only.py \
    --indices-dir data/indices \
    --hybrid-top-k 350 \
    --top-k-chunks 220 \
    --top-k-docs 70 \
    --max-chunks-per-doc 4 \
    --batch-size 16 \
    --disable-pubmed-network \
    --output-name submission.json

In [ ]:
import base64
from IPython.display import HTML, display

zip_path = "/kaggle/working/submission.zip"
with open(zip_path, "rb") as f:
    data_b64 = base64.b64encode(f.read()).decode()

html_button = f'''
<a href="data:application/zip;base64,{data_b64}" download="submission.zip" 
   style="display:inline-block; padding:12px 24px; background:#007acc; color:white; 
          text-decoration:none; border-radius:6px; font-weight:bold; font-size:15px;">
   ⬇ BẤM VÀO ĐÂY ĐỂ TẢI SUBMISSION.ZIP VỀ MÁY
</a>
'''
display(HTML(html_button))

In [ ]:
print("hi")